## Homework 2: Vector Search

1. Embedding a query (1 point)

In [ ]:
from embedder import Embedder

embedder = Embedder(path="models/Xenova/all-MiniLM-L6-v2")

query = "How does approximate nearest neighbor search work?"
v = embedder.encode(query)

# v = embedder.encode(query, normalize=False)  # tried this first, numbers looked way too big

print(v.shape)
print(v[0])

2. Cosine similarity

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

print(len(documents))

72


In [ ]:
doc = next(d for d in documents if d["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md")

# doc = documents[7]  # guessed the index instead of filtering, wrong file lol

doc_vec = embedder.encode(doc["content"])

import numpy as np
similarity = np.dot(v, doc_vec)
print(similarity)

3. Chunking and search by hand

In [5]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(len(chunks))

295


In [6]:
import numpy as np

chunk_texts = [c["content"] for c in chunks]
X = embedder.encode_batch(chunk_texts)

print(X.shape)

scores = X.dot(v)
best_idx = np.argmax(scores)

print(chunks[best_idx]["filename"])
print(scores[best_idx])

(295, 384)
02-vector-search/lessons/07-sqlitesearch-vector.md
0.6489017718578813


4. Vector search with minsearch 

In [ ]:
from minsearch import VectorSearch

vindex = VectorSearch()
vindex.fit(X, chunks)

query4 = "What metric do we use to evaluate a search engine?"
v4 = embedder.encode(query4)

results = vindex.search(v4, num_results=5)
# results = vindex.search(query4, num_results=5)  # forgot to embed the query first, threw an error
print(results[0]["filename"])

5. Text search vs vector search

In [8]:
from minsearch import Index

text_index = Index(text_fields=["content"])
text_index.fit(chunks)

query5 = "How do I store vectors in PostgreSQL?"
v5 = embedder.encode(query5)

vector_results = vindex.search(v5, num_results=5)
keyword_results = text_index.search(query5, num_results=5)

vector_files = [r["filename"] for r in vector_results]
keyword_files = [r["filename"] for r in keyword_results]

print("Vector:", vector_files)
print("Keyword:", keyword_files)

only_in_vector = set(vector_files) - set(keyword_files)
print("Only in vector:", only_in_vector)

Vector: ['02-vector-search/lessons/08-pgvector.md', '02-vector-search/lessons/08-pgvector.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/08-pgvector.md', '02-vector-search/lessons/08-pgvector.md']
Keyword: ['02-vector-search/lessons/02-embeddings.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/01-intro.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/01-intro.md']
Only in vector: {'02-vector-search/lessons/08-pgvector.md'}


6. Hybrid search

In [9]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

query6 = "How do I give the model access to tools?"
v6 = embedder.encode(query6)

vector_results6 = vindex.search(v6, num_results=5)
keyword_results6 = text_index.search(query6, num_results=5)

fused = rrf([vector_results6, keyword_results6])

print(fused[0]["filename"])

01-agentic-rag/lessons/13-function-calling.md
